<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES3_01_ClinVar_XML_Normalizer_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES 3.0 — Notebook 01
## ClinVar XML Normalizer: VCV / RCV / SCV → Immutable Parquet

**Google Colab / GitHub-ready**

This notebook implements **GES 3.0 Stage 01: `clinvar_xml_normalizer`**.

It consumes the Stage 00 canonical ClinVar release manifest and creates a release-partitioned normalized data lake for the later longitudinal linkage, evidence graph, event ontology, and forecasting stages.

### What is new in this stage
Stage 00 established that a monthly longitudinal archive is available. Stage 01 converts those monthly XML releases into a stable analytical representation while preserving:

- VCV identity
- RCV variant-condition identity
- SCV submitted assertions
- submitter provenance
- classification and review metadata
- condition identifiers/names
- gene symbols
- citations/evidence identifiers
- source release, XML generation, and source checksums
- current-versus-legacy XML semantics

### Critical methodological rule


Current-format ClinVar XML can represent separate germline, somatic clinical-impact, and oncogenicity classifications. The legacy format used a single classification representation. This notebook therefore stores the legacy aggregate classification in its own fields and creates only a conservative *germline-candidate* indicator. Final germline cohort eligibility must be frozen later.

### Colab design
The complete ClinVar XML releases are very large. This notebook can **stream a `.xml.gz` release directly from NCBI**, parse it incrementally, write Parquet shards, and discard XML elements as they are processed. It therefore does not need to store every compressed XML release simultaneously.


### Primary outputs per release
- `vcv_state/*.parquet`
- `rcv_state/*.parquet`
- `scv_state/*.parquet`
- `vcv_rcv_link/*.parquet`
- `release_qc.json`
- `release_complete.json` only after a successful full-release parse

### NCBI references
- ClinVar downloads: https://www.ncbi.nlm.nih.gov/clinvar/docs/downloads/
- ClinVar release cycle: https://www.ncbi.nlm.nih.gov/clinvar/docs/release_cycle/
- ClinVar accession paths: https://www.ncbi.nlm.nih.gov/clinvar/docs/identifiers/
- ClinVar review status / XML examples: https://www.ncbi.nlm.nih.gov/clinvar/docs/review_status/
- ClinVar classification types: https://www.ncbi.nlm.nih.gov/clinvar/docs/clinsig/
- ClinVar XML README: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/_README

## 2. Install dependencies

In [1]:
!pip -q install lxml pyarrow pandas requests tqdm beautifulsoup4

## 3. Imports and reproducibility metadata

In [2]:
from __future__ import annotations

import concurrent.futures as cf
import gzip
import hashlib
import io
import json
import os
import platform
import re
import shutil
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import bs4
import lxml
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from lxml import etree
from tqdm.auto import tqdm

RUN_UTC = datetime.now(timezone.utc).isoformat()

print("GES 3.0 Stage 01 started:", RUN_UTC)
print("Python:", sys.version.split()[0])
print("lxml:", lxml.__version__)
print("pyarrow:", pa.__version__)
print("pandas:", pd.__version__)

GES 3.0 Stage 01 started: 2026-08-16T17:24:41.113423+00:00
Python: 3.12.13
lxml: 6.1.1
pyarrow: 18.1.0
pandas: 2.2.2


## 4. Configuration

Three profiles are supported.

### `PILOT_SCHEMA_BRIDGE` — default
- Exercises a legacy month and a current-format month.
- Stops after a limited number of records per source.
- Useful for parser and schema-bridge QC.
- **Does not create a production-complete marker and cannot verify the full compressed-file MD5 because the stream is intentionally stopped early.**

### `PRODUCTION_BATCH`
- Full parse for a user-selected month range.
- No record limit.
- Requires the frozen Stage 00 manifest.
- Verifies NCBI MD5 after the complete stream.
- Writes completion markers so reruns can skip already completed releases.

### `FULL_LONGITUDINAL`
- Selects all canonical Stage 00 months.
- Intended for repeated/resumable Colab sessions, not a single uninterrupted notebook session.

**Recommended workflow:** pilot → inspect QC → production batches of a few months at a time → combine later.

In [3]:
# ============================================================
# USER CONFIGURATION
# ============================================================

RUN_PROFILE = "PILOT_SCHEMA_BRIDGE"
# Options:
#   "PILOT_SCHEMA_BRIDGE"
#   "PRODUCTION_BATCH"
#   "FULL_LONGITUDINAL"

# Stage 00 manifest.
# AUTO searches common local/Drive locations and uploaded ZIP/CSV.
# For production, this notebook refuses to proceed without a frozen Stage 00 manifest.
MANIFEST_MODE = "AUTO"

# Pilot settings: choose one likely legacy month and one current month.
PILOT_MONTHS = ["2023-01", "2026-08"]
PILOT_RECORD_LIMIT_PER_SOURCE = 50_000

# Production batch settings.
BATCH_START_MONTH = "2021-01"
BATCH_END_MONTH = "2021-03"

# Parse both data products.
MODELS_TO_PROCESS = ["VCV", "RCV"]

# Output storage.
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_ROOT = "/content/drive/MyDrive/GES3"
LOCAL_ROOT = "/content/GES3"

# Streaming is the Colab-friendly default.
# It avoids keeping multi-GB compressed XML files on local disk.
STREAM_REMOTE_GZIP = True

# Number of normalized rows written per Parquet shard.
PARQUET_ROWS_PER_SHARD = 50_000

# HTTP behavior.
REQUEST_TIMEOUT = 120
HTTP_CHUNK_SIZE = 1024 * 1024
USER_AGENT = "GES3-ClinVar-Normalizer/1.0 research pipeline"

# Retry the whole source stream on transient failure.
MAX_SOURCE_ATTEMPTS = 3
RETRY_BACKOFF_SECONDS = 8

# If True, a production release that already has release_complete.json is skipped.
RESUME_COMPLETED_RELEASES = True

# Safety: never automatically delete a completed release partition.
DELETE_INCOMPLETE_PARTITION_BEFORE_RETRY = True

print("RUN_PROFILE =", RUN_PROFILE)
print("MODELS_TO_PROCESS =", MODELS_TO_PROCESS)

RUN_PROFILE = PILOT_SCHEMA_BRIDGE
MODELS_TO_PROCESS = ['VCV', 'RCV']


## 5. Optional Google Drive mount

For the default pilot, local Colab storage is fine. For production batches, Drive is recommended because normalized Parquet outputs should persist across Colab sessions.

In [4]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(GOOGLE_DRIVE_ROOT)
else:
    ROOT = Path(LOCAL_ROOT)

STAGE00_DIR = ROOT / "stage00"
STAGE01_DIR = ROOT / "stage01"
DATA_DIR = STAGE01_DIR / "normalized"
QC_DIR = STAGE01_DIR / "qc"
META_DIR = STAGE01_DIR / "metadata"

for d in [STAGE00_DIR, STAGE01_DIR, DATA_DIR, QC_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Stage 01 output:", STAGE01_DIR)

ROOT: /content/GES3
Stage 01 output: /content/GES3/stage01


## 6. Load the frozen Stage 00 manifest

The preferred inputs are either:
- `clinvar_release_manifest_canonical.csv`, or
- `GES3_STAGE00_ARTIFACTS.zip`.

The cell searches local Colab paths and Drive paths first.

For the **pilot only**, if no Stage 00 artifact is available, the notebook can reconstruct the same canonical inventory logic directly from NCBI. A production run requires a frozen Stage 00 manifest so the source list cannot drift silently.

In [5]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})

def find_stage00_artifact():
    candidates = [
        Path("/content/clinvar_release_manifest_canonical.csv"),
        Path("/content/ges3_stage00/clinvar_release_manifest_canonical.csv"),
        STAGE00_DIR / "clinvar_release_manifest_canonical.csv",
        Path("/content/GES3_STAGE00_ARTIFACTS.zip"),
        STAGE00_DIR / "GES3_STAGE00_ARTIFACTS.zip",
    ]

    for p in candidates:
        if p.exists():
            return p

    # Also inspect top-level /content uploads.
    for p in Path("/content").glob("*STAGE00*.zip"):
        return p
    for p in Path("/content").glob("*release_manifest_canonical*.csv"):
        return p

    return None

def load_manifest_from_artifact(path: Path) -> tuple[pd.DataFrame, str]:
    path = Path(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path), f"frozen_csv:{path}"

    if path.suffix.lower() == ".zip":
        extract_dir = STAGE00_DIR / "imported_stage00"
        extract_dir.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(path), str(extract_dir))

        hits = list(extract_dir.rglob("clinvar_release_manifest_canonical.csv"))
        if not hits:
            raise FileNotFoundError(
                "ZIP did not contain clinvar_release_manifest_canonical.csv"
            )
        return pd.read_csv(hits[0]), f"frozen_zip:{path}"

    raise ValueError(f"Unsupported Stage 00 artifact: {path}")

stage00_artifact = find_stage00_artifact()
manifest = None
manifest_origin = None

if stage00_artifact is not None:
    manifest, manifest_origin = load_manifest_from_artifact(stage00_artifact)
    print("Loaded Stage 00 artifact:", stage00_artifact)
else:
    print("No frozen Stage 00 manifest found locally.")

No frozen Stage 00 manifest found locally.


## 7. Pilot-only fallback: reconstruct the Stage 00 canonical manifest

This fallback exists so the parser can be tested in a fresh Colab session. It is **not allowed for production**.

The canonical policy is the same as Stage 00:
1. discover monthly VCV/RCV files;
2. discover legacy and current archive families;
3. prefer current XML when the same model/month exists in both formats;
4. otherwise use legacy.

In [6]:
BASE = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/"

SOURCE_REGISTRY = pd.DataFrame([
    {
        "data_model": "VCV",
        "format_generation": "current",
        "root_url": BASE,
        "archive_url": BASE + "archive/",
        "filename_prefix": "ClinVarVCVRelease_",
    },
    {
        "data_model": "RCV",
        "format_generation": "current",
        "root_url": BASE + "RCV_release/",
        "archive_url": BASE + "RCV_release/archive/",
        "filename_prefix": "ClinVarRCVRelease_",
    },
    {
        "data_model": "VCV",
        "format_generation": "legacy",
        "root_url": BASE + "VCV_xml_old_format/",
        "archive_url": BASE + "VCV_xml_old_format/archive/",
        "filename_prefix": "ClinVarVariationRelease_",
    },
    {
        "data_model": "RCV",
        "format_generation": "legacy",
        "root_url": BASE + "RCV_xml_old_format/",
        "archive_url": BASE + "RCV_xml_old_format/archive/",
        "filename_prefix": "ClinVarFullRelease_",
    },
])

def get_text(url: str) -> str:
    r = SESSION.get(url, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.text

def list_apache_links(url: str) -> pd.DataFrame:
    soup = BeautifulSoup(get_text(url), "html.parser")
    rows = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href in ("../", "/"):
            continue
        rows.append({
            "name": a.get_text(" ", strip=True) or href,
            "href": href,
            "url": urljoin(url, href),
        })
    return pd.DataFrame(rows)

def discover_archive_years(archive_url: str) -> list[int]:
    try:
        idx = list_apache_links(archive_url)
    except Exception:
        return []
    years = []
    for href in idx.get("href", []):
        m = re.fullmatch(r"(\d{4})/", str(href))
        if m:
            years.append(int(m.group(1)))
    return sorted(set(years))

def parse_release_month(name: str) -> str | None:
    m = re.search(r"_(\d{4})-(\d{2})\.xml\.gz$", str(name))
    return f"{m.group(1)}-{m.group(2)}" if m else None

def discover_source(row: pd.Series) -> pd.DataFrame:
    prefix = row["filename_prefix"]
    pattern = re.compile(re.escape(prefix) + r"\d{4}-\d{2}\.xml\.gz$")
    frames = []

    try:
        root = list_apache_links(row["root_url"])
        root["scope"] = "root"
        frames.append(root)
    except Exception:
        pass

    for year in discover_archive_years(row["archive_url"]):
        if year < 2021:
            continue
        u = urljoin(row["archive_url"], f"{year}/")
        try:
            d = list_apache_links(u)
            d["scope"] = f"archive/{year}"
            frames.append(d)
        except Exception:
            pass

    if not frames:
        return pd.DataFrame()

    x = pd.concat(frames, ignore_index=True)
    x = x[x["name"].map(lambda n: bool(pattern.fullmatch(str(n))))].copy()
    x["data_model"] = row["data_model"]
    x["format_generation"] = row["format_generation"]
    x["release_month"] = x["name"].map(parse_release_month)
    x["md5_url"] = x["url"] + ".md5"
    return x

def fetch_md5(md5_url: str) -> str | None:
    try:
        text = get_text(md5_url)
        m = re.search(r"\b([a-fA-F0-9]{32})\b", text)
        return m.group(1).lower() if m else None
    except Exception:
        return None

def reconstruct_canonical_manifest() -> pd.DataFrame:
    parts = []
    for _, row in SOURCE_REGISTRY.iterrows():
        p = discover_source(row)
        if not p.empty:
            parts.append(p)

    if not parts:
        raise RuntimeError("Could not reconstruct ClinVar archive manifest.")

    x = pd.concat(parts, ignore_index=True).drop_duplicates(
        ["data_model", "format_generation", "release_month", "url"]
    )
    x["priority"] = x["format_generation"].map({"current": 0, "legacy": 1})
    x = x.sort_values(["data_model", "release_month", "priority", "url"])
    x = x.groupby(["data_model", "release_month"], as_index=False).head(1).copy()

    # Fetch MD5 only for rows likely to be used in the pilot.
    wanted = set(PILOT_MONTHS)
    mask = x["release_month"].isin(wanted)
    for idx in x.index[mask]:
        x.loc[idx, "ncbi_md5"] = fetch_md5(x.loc[idx, "md5_url"])

    return x.sort_values(["release_month", "data_model"]).reset_index(drop=True)

if manifest is None:
    if RUN_PROFILE != "PILOT_SCHEMA_BRIDGE":
        raise RuntimeError(
            "Production requires the frozen Stage 00 canonical manifest. "
            "Upload clinvar_release_manifest_canonical.csv or GES3_STAGE00_ARTIFACTS.zip."
        )

    manifest = reconstruct_canonical_manifest()
    manifest_origin = "reconstructed_live_for_pilot_only"
    print("WARNING: using live-reconstructed manifest for PILOT only.")

manifest["release_month"] = manifest["release_month"].astype(str)
manifest["data_model"] = manifest["data_model"].astype(str).str.upper()
manifest["format_generation"] = (
    manifest["format_generation"].astype(str).str.lower()
)

required_manifest_cols = {
    "release_month", "data_model", "format_generation", "url"
}
missing = required_manifest_cols - set(manifest.columns)
if missing:
    raise ValueError(f"Manifest missing required columns: {sorted(missing)}")

print("Manifest origin:", manifest_origin)
print("Manifest rows:", len(manifest))
display(manifest.head())

Manifest origin: reconstructed_live_for_pilot_only
Manifest rows: 136


,name,href,url,scope,data_model,format_generation,release_month,md5_url,priority,ncbi_md5
0,ClinVarFullRelease_2021-01.xml.gz,ClinVarFullRelease_2021-01.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,archive/2021,RCV,legacy,2021-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,1,NaN
1,ClinVarVariationRelease_2021-01.xml.gz,ClinVarVariationRelease_2021-01.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,archive/2021,VCV,legacy,2021-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,1,NaN
2,ClinVarFullRelease_2021-02.xml.gz,ClinVarFullRelease_2021-02.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,archive/2021,RCV,legacy,2021-02,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,1,NaN
3,ClinVarVariationRelease_2021-02.xml.gz,ClinVarVariationRelease_2021-02.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,archive/2021,VCV,legacy,2021-02,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,1,NaN
4,ClinVarFullRelease_2021-03.xml.gz,ClinVarFullRelease_2021-03.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,archive/2021,RCV,legacy,2021-03,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,1,NaN


## 8. Select releases for this run

The selected source list is frozen into a Stage 01 run manifest before parsing begins.

In [7]:
def month_between(s: pd.Series, start: str, end: str) -> pd.Series:
    p = pd.PeriodIndex(s.astype(str), freq="M")
    return (p >= pd.Period(start, freq="M")) & (p <= pd.Period(end, freq="M"))

if RUN_PROFILE == "PILOT_SCHEMA_BRIDGE":
    selected = manifest[
        manifest["release_month"].isin(PILOT_MONTHS)
        & manifest["data_model"].isin(MODELS_TO_PROCESS)
    ].copy()
    record_limit = PILOT_RECORD_LIMIT_PER_SOURCE

elif RUN_PROFILE == "PRODUCTION_BATCH":
    selected = manifest[
        month_between(
            manifest["release_month"],
            BATCH_START_MONTH,
            BATCH_END_MONTH,
        )
        & manifest["data_model"].isin(MODELS_TO_PROCESS)
    ].copy()
    record_limit = None

elif RUN_PROFILE == "FULL_LONGITUDINAL":
    selected = manifest[
        manifest["data_model"].isin(MODELS_TO_PROCESS)
    ].copy()
    record_limit = None

else:
    raise ValueError(f"Unknown RUN_PROFILE: {RUN_PROFILE}")

selected = selected.sort_values(
    ["release_month", "data_model"]
).reset_index(drop=True)

if selected.empty:
    raise RuntimeError("No manifest rows selected for this run.")

run_manifest_path = META_DIR / (
    f"stage01_run_manifest_{RUN_PROFILE}_{RUN_UTC[:10]}.csv"
)
selected.to_csv(run_manifest_path, index=False)

print("Selected source files:", len(selected))
print("Record limit/source:", record_limit)
display(selected[[
    c for c in [
        "release_month", "data_model", "format_generation",
        "url", "ncbi_md5"
    ] if c in selected.columns
]])

Selected source files: 4
Record limit/source: 50000


,release_month,data_model,format_generation,url,ncbi_md5
0,2023-01,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,d00be8862bbbd1d5a8b991c30246d224
1,2023-01,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,97acfb6a750b6a054c9800ac9f9a9924
2,2026-08,RCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,f6d225ba092e3d79b68dfaf06f6deb67
3,2026-08,VCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,15eb690d6f3c9845373e53f7e9ad83f0


## 9. Normalized table contract

Stage 01 deliberately writes a **stable, explicit schema** rather than arbitrary flattened XML.

### `vcv_state`
One VCV aggregate state per release month.

### `rcv_state`
One RCV variant-condition aggregate state per release month. This becomes the primary GES 3.0 prediction-unit foundation.

### `scv_state`
One submitted assertion attached to its parent RCV or VCV record for that release. RCV-derived SCVs are especially important because they retain the variant-condition context.

### `vcv_rcv_link`
Links a VCV aggregate record to RCV accessions visible in that release.

JSON strings are used only for repeated provenance fields (genes, conditions, citations, RCV lists). Later notebooks can explode these into graph tables.

In [8]:
GERMLINE_TERMS = {
    "benign",
    "likely benign",
    "uncertain significance",
    "vus-high",
    "vus-mid",
    "vus-low",
    "likely pathogenic",
    "pathogenic",
    "likely pathogenic, low penetrance",
    "pathogenic, low penetrance",
    "uncertain risk allele",
    "likely risk allele",
    "established risk allele",
    "drug response",
    "association",
    "protective",
    "affects",
    "conflicting data from submitters",
    "conflicting classifications of pathogenicity",
    "other",
    "not provided",
}

VCV_COLUMNS = [
    "release_month", "source_url", "source_format",
    "vcv_accession", "vcv_version", "variation_id",
    "variation_type", "variation_name",
    "date_created", "date_last_updated",
    "germline_description", "germline_review_status",
    "germline_date_last_evaluated",
    "somatic_clinical_impact_description",
    "somatic_clinical_impact_review_status",
    "oncogenicity_description", "oncogenicity_review_status",
    "legacy_classification_description",
    "legacy_review_status", "legacy_date_last_evaluated",
    "legacy_germline_candidate",
    "gene_symbols_json", "rcv_accessions_json",
    "citation_ids_json", "scv_count",
]

RCV_COLUMNS = [
    "release_month", "source_url", "source_format",
    "rcv_accession", "rcv_version",
    "vcv_accession", "vcv_version",
    "variation_id", "variation_name",
    "germline_description", "germline_review_status",
    "germline_date_last_evaluated",
    "somatic_clinical_impact_description",
    "somatic_clinical_impact_review_status",
    "oncogenicity_description", "oncogenicity_review_status",
    "legacy_classification_description",
    "legacy_review_status", "legacy_date_last_evaluated",
    "legacy_germline_candidate",
    "condition_names_json", "condition_ids_json",
    "gene_symbols_json", "citation_ids_json",
    "scv_count",
]

SCV_COLUMNS = [
    "release_month", "source_url", "source_format",
    "source_data_model",
    "parent_vcv_accession", "parent_rcv_accession",
    "scv_accession", "scv_version",
    "classification_type_raw",
    "classification_description",
    "review_status", "date_last_evaluated",
    "submitter_name", "submitter_org_id",
    "submitter_local_key", "submitter_date",
    "assertion_method",
    "collection_method", "origin",
    "condition_names_json", "condition_ids_json",
    "citation_ids_json",
    "germline_term_candidate",
]

VCV_RCV_LINK_COLUMNS = [
    "release_month", "source_url", "source_format",
    "vcv_accession", "vcv_version",
    "rcv_accession", "rcv_version",
]

TABLE_COLUMNS = {
    "vcv_state": VCV_COLUMNS,
    "rcv_state": RCV_COLUMNS,
    "scv_state": SCV_COLUMNS,
    "vcv_rcv_link": VCV_RCV_LINK_COLUMNS,
}

print("Normalized schema contract loaded.")
for name, cols in TABLE_COLUMNS.items():
    print(name, "columns:", len(cols))

Normalized schema contract loaded.
vcv_state columns: 25
rcv_state columns: 25
scv_state columns: 23
vcv_rcv_link columns: 7


## 10. XML utility functions

These functions are namespace-tolerant and intentionally conservative. They look for known ClinVar semantic elements while retaining raw text when exact historical representation differs.

In [9]:
def localname(tag) -> str:
    if not isinstance(tag, str):
        return ""
    if "}" in tag:
        return tag.rsplit("}", 1)[-1]
    return tag

def clean_text(x):
    if x is None:
        return None
    x = re.sub(r"\s+", " ", str(x)).strip()
    return x if x else None

def iter_desc(elem, names=None):
    names = set(names) if names else None
    for x in elem.iter():
        n = localname(x.tag)
        if names is None or n in names:
            yield x

def first_desc(elem, names):
    names = set(names)
    for x in elem.iter():
        if localname(x.tag) in names:
            return x
    return None

def first_text(elem, names):
    x = first_desc(elem, names)
    return clean_text(x.text) if x is not None else None

def attr_any(elem, names):
    if elem is None:
        return None
    for k in names:
        if k in elem.attrib:
            return clean_text(elem.attrib.get(k))
    # Case-insensitive fallback.
    lower = {str(k).lower(): v for k, v in elem.attrib.items()}
    for k in names:
        if str(k).lower() in lower:
            return clean_text(lower[str(k).lower()])
    return None

def child_text(container, preferred_names):
    if container is None:
        return None
    for x in container.iter():
        if localname(x.tag) in preferred_names:
            t = clean_text(x.text)
            if t:
                return t
    return None

def json_list(values):
    vals = []
    seen = set()
    for v in values:
        v = clean_text(v)
        if v and v not in seen:
            vals.append(v)
            seen.add(v)
    return json.dumps(vals, ensure_ascii=False)

def extract_element_values(elem, preferred_type=None):
    out = []
    for x in iter_desc(elem, {"ElementValue"}):
        typ = attr_any(x, ["Type"])
        if preferred_type is None or (
            typ and typ.lower() == preferred_type.lower()
        ):
            t = clean_text(x.text)
            if t:
                out.append(t)
    return out

def extract_citations(elem):
    values = []
    for cit in iter_desc(elem, {"Citation"}):
        for x in cit.iter():
            n = localname(x.tag)
            if n in {"ID", "CitationText", "URL"}:
                t = clean_text(x.text)
                if not t:
                    continue
                src = attr_any(x, ["Source", "Type"])
                values.append(f"{src}:{t}" if src else t)
    return values

def extract_conditions(elem):
    names = []
    ids = []

    for trait in iter_desc(elem, {"Trait"}):
        preferred = []
        fallback = []

        for name_el in trait.iter():
            if localname(name_el.tag) != "Name":
                continue
            vals = extract_element_values(name_el, preferred_type="Preferred")
            preferred.extend(vals)
            if not vals:
                fallback.extend(extract_element_values(name_el))

        names.extend(preferred or fallback)

        for xref in iter_desc(trait, {"XRef"}):
            db = attr_any(xref, ["DB"])
            xid = attr_any(xref, ["ID"])
            if db and xid:
                ids.append(f"{db}:{xid}")

    return names, ids

def extract_gene_symbols(elem):
    symbols = []

    for gene in iter_desc(elem, {"Gene"}):
        for sym in iter_desc(gene, {"Symbol"}):
            vals = extract_element_values(sym, preferred_type="Preferred")
            if not vals:
                vals = extract_element_values(sym)
            symbols.extend(vals)

        # Some schema generations expose Symbol as an attribute.
        s = attr_any(gene, ["Symbol"])
        if s:
            symbols.append(s)

    return symbols

def classification_fields(container):
    if container is None:
        return {
            "description": None,
            "review_status": None,
            "date_last_evaluated": None,
        }

    description = child_text(
        container,
        {"Description", "ClinicalSignificanceDescription"},
    )
    review_status = child_text(container, {"ReviewStatus"})
    date_last_evaluated = (
        attr_any(container, ["DateLastEvaluated"])
        or child_text(container, {"DateLastEvaluated"})
    )

    return {
        "description": description,
        "review_status": review_status,
        "date_last_evaluated": date_last_evaluated,
    }

def find_named_container(elem, name):
    for x in elem.iter():
        if localname(x.tag) == name:
            return x
    return None

def current_aggregate_classifications(elem):
    germline = classification_fields(
        find_named_container(elem, "GermlineClassification")
    )
    somatic = classification_fields(
        find_named_container(elem, "SomaticClinicalImpact")
    )
    oncogenic = classification_fields(
        find_named_container(elem, "OncogenicityClassification")
    )

    return germline, somatic, oncogenic

def legacy_aggregate_classification(elem):
    # RCV legacy commonly uses ClinicalSignificance.
    cs = find_named_container(elem, "ClinicalSignificance")
    if cs is not None:
        return classification_fields(cs)

    # VCV legacy aggregate data are represented within InterpretedRecord.
    ir = find_named_container(elem, "InterpretedRecord")
    if ir is not None:
        # Search a narrower interpretation/classification container first.
        for name in ["Interpretation", "Classification"]:
            c = find_named_container(ir, name)
            if c is not None:
                f = classification_fields(c)
                if any(f.values()):
                    return f
        return classification_fields(ir)

    return {
        "description": None,
        "review_status": None,
        "date_last_evaluated": None,
    }

def is_germline_candidate(description):
    if not description:
        return False
    d = clean_text(description).lower()
    if d in GERMLINE_TERMS:
        return True
    # Aggregate conflict strings can include extra wording.
    if "conflicting classifications of pathogenicity" in d:
        return True
    return False

def clear_element(elem):
    elem.clear()
    parent = elem.getparent()
    if parent is not None:
        while elem.getprevious() is not None:
            del parent[0]

print("XML utilities loaded.")

XML utilities loaded.


## 11. Parse VCV aggregate records

The VCV accession is taken from `VariationArchive/@Accession`, consistent with NCBI's identifier documentation. RCV links are taken from `RCVList/RCVAccession`.

Current-format aggregate classification axes are retained separately. Legacy single-classification fields remain explicitly marked as legacy.

In [10]:
def parse_vcv_record(elem, release_month, source_url, source_format):
    vcv_accession = attr_any(elem, ["Accession"])
    if not (vcv_accession and vcv_accession.startswith("VCV")):
        return None, [], []

    vcv_version = attr_any(elem, ["Version"])
    variation_id = attr_any(elem, ["VariationID", "VariationId", "ID"])
    variation_type = attr_any(elem, ["VariationType", "Type"])

    variation_name = (
        attr_any(elem, ["VariationName"])
        or first_text(elem, {"VariationName"})
        or first_text(elem, {"Name"})
    )

    date_created = attr_any(elem, ["DateCreated"])
    date_last_updated = attr_any(
        elem, ["DateLastUpdated", "DateLastUpdate"]
    )

    if source_format == "current":
        germline, somatic, oncogenic = current_aggregate_classifications(elem)
        legacy = {
            "description": None,
            "review_status": None,
            "date_last_evaluated": None,
        }
    else:
        germline = {
            "description": None,
            "review_status": None,
            "date_last_evaluated": None,
        }
        somatic = dict(germline)
        oncogenic = dict(germline)
        legacy = legacy_aggregate_classification(elem)

    genes = extract_gene_symbols(elem)
    citations = extract_citations(elem)

    rcv_links = []
    rcv_accessions = []

    for x in iter_desc(elem, {"RCVAccession"}):
        acc = attr_any(x, ["Accession", "Acc"])
        ver = attr_any(x, ["Version"])
        if acc and acc.startswith("RCV"):
            rcv_accessions.append(acc)
            rcv_links.append({
                "release_month": release_month,
                "source_url": source_url,
                "source_format": source_format,
                "vcv_accession": vcv_accession,
                "vcv_version": vcv_version,
                "rcv_accession": acc,
                "rcv_version": ver,
            })

    scvs = parse_scv_children(
        elem=elem,
        source_data_model="VCV",
        parent_vcv_accession=vcv_accession,
        parent_rcv_accession=None,
        release_month=release_month,
        source_url=source_url,
        source_format=source_format,
    )

    row = {
        "release_month": release_month,
        "source_url": source_url,
        "source_format": source_format,
        "vcv_accession": vcv_accession,
        "vcv_version": vcv_version,
        "variation_id": variation_id,
        "variation_type": variation_type,
        "variation_name": variation_name,
        "date_created": date_created,
        "date_last_updated": date_last_updated,

        "germline_description": germline["description"],
        "germline_review_status": germline["review_status"],
        "germline_date_last_evaluated": germline["date_last_evaluated"],

        "somatic_clinical_impact_description": somatic["description"],
        "somatic_clinical_impact_review_status": somatic["review_status"],

        "oncogenicity_description": oncogenic["description"],
        "oncogenicity_review_status": oncogenic["review_status"],

        "legacy_classification_description": legacy["description"],
        "legacy_review_status": legacy["review_status"],
        "legacy_date_last_evaluated": legacy["date_last_evaluated"],
        "legacy_germline_candidate": is_germline_candidate(
            legacy["description"]
        ),

        "gene_symbols_json": json_list(genes),
        "rcv_accessions_json": json_list(rcv_accessions),
        "citation_ids_json": json_list(citations),
        "scv_count": len(scvs),
    }

    return row, rcv_links, scvs

## 12. Parse SCV submitted assertions

NCBI documents different SCV accession attribute names across the VCV and RCV XML products:
- VCV: `ClinicalAssertion/ClinVarAccession/@Accession`
- RCV: `ClinVarAssertion/ClinVarAccession/@Acc`

The parser handles both while preserving the source data model.

In [11]:
def infer_submission_meta(assertion):
    submitter_name = None
    submitter_org_id = None
    submitter_local_key = None
    submitter_date = None

    # ClinVarSubmissionID is present in multiple schema generations.
    sid = find_named_container(assertion, "ClinVarSubmissionID")
    if sid is not None:
        submitter_name = attr_any(
            sid, ["submitter", "Submitter", "SubmitterName"]
        )
        submitter_org_id = attr_any(
            sid, ["OrgID", "orgID", "SubmitterID"]
        )
        submitter_local_key = attr_any(
            sid, ["localKey", "LocalKey", "submitterKey"]
        )
        submitter_date = attr_any(
            sid, ["submitterDate", "SubmitterDate", "Date"]
        )

    # Fallbacks for newer representations.
    if submitter_name is None:
        for x in assertion.iter():
            n = localname(x.tag)
            if n in {"Submitter", "Organization"}:
                submitter_name = (
                    attr_any(x, ["Name", "name", "SubmitterName"])
                    or clean_text(x.text)
                )
                submitter_org_id = submitter_org_id or attr_any(
                    x, ["OrgID", "ID", "SubmitterID"]
                )
                if submitter_name:
                    break

    return (
        submitter_name,
        submitter_org_id,
        submitter_local_key,
        submitter_date,
    )

def infer_assertion_classification(assertion, source_format):
    # Current VCV/RCV SCVs use Classification.
    c = find_named_container(assertion, "Classification")
    if c is not None:
        f = classification_fields(c)
        ctype = attr_any(c, ["Type", "ClassificationType"])
        return ctype, f

    # Legacy RCV commonly uses ClinicalSignificance.
    c = find_named_container(assertion, "ClinicalSignificance")
    if c is not None:
        return "legacy_single", classification_fields(c)

    # Legacy VCV may expose interpretation data under the assertion.
    c = find_named_container(assertion, "Interpretation")
    if c is not None:
        return "legacy_single", classification_fields(c)

    # Last resort: keep review/date if directly nested.
    f = classification_fields(assertion)
    return (
        "legacy_single" if source_format == "legacy" else None,
        f,
    )

def extract_assertion_method(assertion):
    # Prefer explicit method text.
    for name in [
        "AssertionMethod",
        "Method",
        "Description",
    ]:
        x = find_named_container(assertion, name)
        if x is not None:
            t = clean_text(x.text)
            if t:
                return t
    return None

def parse_one_scv(
    assertion,
    source_data_model,
    parent_vcv_accession,
    parent_rcv_accession,
    release_month,
    source_url,
    source_format,
):
    acc_el = find_named_container(assertion, "ClinVarAccession")
    if acc_el is None:
        return None

    scv_accession = attr_any(acc_el, ["Accession", "Acc"])
    if not (scv_accession and scv_accession.startswith("SCV")):
        return None

    scv_version = attr_any(acc_el, ["Version"])
    ctype, cf = infer_assertion_classification(assertion, source_format)

    (
        submitter_name,
        submitter_org_id,
        submitter_local_key,
        submitter_date,
    ) = infer_submission_meta(assertion)

    condition_names, condition_ids = extract_conditions(assertion)
    citations = extract_citations(assertion)

    collection_method = first_text(
        assertion,
        {"MethodType", "CollectionMethod"},
    )
    origin = first_text(
        assertion,
        {"Origin", "AlleleOrigin"},
    )
    assertion_method = extract_assertion_method(assertion)

    description = cf["description"]

    return {
        "release_month": release_month,
        "source_url": source_url,
        "source_format": source_format,
        "source_data_model": source_data_model,
        "parent_vcv_accession": parent_vcv_accession,
        "parent_rcv_accession": parent_rcv_accession,
        "scv_accession": scv_accession,
        "scv_version": scv_version,
        "classification_type_raw": ctype,
        "classification_description": description,
        "review_status": cf["review_status"],
        "date_last_evaluated": cf["date_last_evaluated"],
        "submitter_name": submitter_name,
        "submitter_org_id": submitter_org_id,
        "submitter_local_key": submitter_local_key,
        "submitter_date": submitter_date,
        "assertion_method": assertion_method,
        "collection_method": collection_method,
        "origin": origin,
        "condition_names_json": json_list(condition_names),
        "condition_ids_json": json_list(condition_ids),
        "citation_ids_json": json_list(citations),
        "germline_term_candidate": is_germline_candidate(description),
    }

def parse_scv_children(
    elem,
    source_data_model,
    parent_vcv_accession,
    parent_rcv_accession,
    release_month,
    source_url,
    source_format,
):
    assertion_tag = (
        "ClinicalAssertion" if source_data_model == "VCV"
        else "ClinVarAssertion"
    )

    rows = []
    for assertion in iter_desc(elem, {assertion_tag}):
        row = parse_one_scv(
            assertion=assertion,
            source_data_model=source_data_model,
            parent_vcv_accession=parent_vcv_accession,
            parent_rcv_accession=parent_rcv_accession,
            release_month=release_month,
            source_url=source_url,
            source_format=source_format,
        )
        if row is not None:
            rows.append(row)

    return rows

print("SCV parser loaded.")

SCV parser loaded.


## 13. Parse RCV aggregate records

The RCV accession comes from `ReferenceClinVarAssertion/ClinVarAccession`, while VCV identity can be represented through `MeasureSet/@Acc` and Variation ID through `MeasureSet/@ID`, consistent with NCBI's accession documentation.

The RCV-level condition and SCV relationship are preserved because the primary future GES 3.0 prediction unit is the **RCV / variant-condition monthly state**.

In [12]:
def parse_rcv_record(elem, release_month, source_url, source_format):
    ref = find_named_container(elem, "ReferenceClinVarAssertion")
    if ref is None:
        return None, []

    acc_el = find_named_container(ref, "ClinVarAccession")
    rcv_accession = attr_any(acc_el, ["Acc", "Accession"])
    if not (rcv_accession and rcv_accession.startswith("RCV")):
        return None, []

    rcv_version = attr_any(acc_el, ["Version"])

    measure_set = find_named_container(ref, "MeasureSet")
    variation_id = attr_any(measure_set, ["ID", "VariationID"])
    vcv_accession = attr_any(measure_set, ["Acc", "Accession"])
    vcv_version = attr_any(measure_set, ["Version"])

    variation_name = (
        first_text(measure_set, {"Name", "ElementValue"})
        if measure_set is not None
        else None
    )

    if source_format == "current":
        germline, somatic, oncogenic = current_aggregate_classifications(ref)
        legacy = {
            "description": None,
            "review_status": None,
            "date_last_evaluated": None,
        }
    else:
        germline = {
            "description": None,
            "review_status": None,
            "date_last_evaluated": None,
        }
        somatic = dict(germline)
        oncogenic = dict(germline)
        legacy = legacy_aggregate_classification(ref)

    condition_names, condition_ids = extract_conditions(ref)
    genes = extract_gene_symbols(ref)
    citations = extract_citations(ref)

    scvs = parse_scv_children(
        elem=elem,
        source_data_model="RCV",
        parent_vcv_accession=vcv_accession,
        parent_rcv_accession=rcv_accession,
        release_month=release_month,
        source_url=source_url,
        source_format=source_format,
    )

    row = {
        "release_month": release_month,
        "source_url": source_url,
        "source_format": source_format,
        "rcv_accession": rcv_accession,
        "rcv_version": rcv_version,
        "vcv_accession": vcv_accession,
        "vcv_version": vcv_version,
        "variation_id": variation_id,
        "variation_name": variation_name,

        "germline_description": germline["description"],
        "germline_review_status": germline["review_status"],
        "germline_date_last_evaluated": germline["date_last_evaluated"],

        "somatic_clinical_impact_description": somatic["description"],
        "somatic_clinical_impact_review_status": somatic["review_status"],

        "oncogenicity_description": oncogenic["description"],
        "oncogenicity_review_status": oncogenic["review_status"],

        "legacy_classification_description": legacy["description"],
        "legacy_review_status": legacy["review_status"],
        "legacy_date_last_evaluated": legacy["date_last_evaluated"],
        "legacy_germline_candidate": is_germline_candidate(
            legacy["description"]
        ),

        "condition_names_json": json_list(condition_names),
        "condition_ids_json": json_list(condition_ids),
        "gene_symbols_json": json_list(genes),
        "citation_ids_json": json_list(citations),
        "scv_count": len(scvs),
    }

    return row, scvs

## 14. Parquet shard writer

Rows are written in bounded batches. No release needs to reside fully in RAM.

In [13]:
class ShardWriter:
    def __init__(self, release_dir: Path, table_name: str, columns: list[str]):
        self.dir = release_dir / table_name
        self.dir.mkdir(parents=True, exist_ok=True)
        self.table_name = table_name
        self.columns = columns
        self.buffer = []
        self.part = 0
        self.total_rows = 0

    def add(self, row):
        if row is None:
            return
        normalized = {c: row.get(c) for c in self.columns}
        self.buffer.append(normalized)
        if len(self.buffer) >= PARQUET_ROWS_PER_SHARD:
            self.flush()

    def add_many(self, rows):
        for row in rows:
            self.add(row)

    def flush(self):
        if not self.buffer:
            return

        df = pd.DataFrame(self.buffer, columns=self.columns)

        # Keep string-like schema stable while preserving boolean/numeric fields.
        table = pa.Table.from_pandas(df, preserve_index=False)
        path = self.dir / f"part-{self.part:05d}.parquet"
        pq.write_table(
            table,
            path,
            compression="zstd",
            use_dictionary=True,
        )

        self.total_rows += len(self.buffer)
        self.part += 1
        self.buffer.clear()

    def close(self):
        self.flush()

def make_writers(release_dir: Path):
    return {
        name: ShardWriter(release_dir, name, cols)
        for name, cols in TABLE_COLUMNS.items()
    }

print("Parquet shard writer loaded.")

Parquet shard writer loaded.


## 15. Streaming compressed-byte hashing

For a **full** source parse, the compressed-byte MD5 is calculated while NCBI's gzip stream is being consumed. At end of stream it is compared with the Stage 00/NCBI MD5.

For a record-limited pilot, the stream ends early by design, so the source hash is marked `partial_stream` and is **not** treated as verified.

In [14]:
class HashingReader:
    def __init__(self, raw):
        self.raw = raw
        self.md5 = hashlib.md5()
        self.sha256 = hashlib.sha256()
        self.bytes_read = 0

    def read(self, size=-1):
        chunk = self.raw.read(size)
        if chunk:
            self.md5.update(chunk)
            self.sha256.update(chunk)
            self.bytes_read += len(chunk)
        return chunk

    def readable(self):
        return True

    def __getattr__(self, name):
        return getattr(self.raw, name)

def source_expected_md5(row):
    val = row.get("ncbi_md5")
    if pd.isna(val) if not isinstance(val, str) else False:
        return None
    val = clean_text(val)
    return val.lower() if val else None

print("Streaming hash wrapper loaded.")

Streaming hash wrapper loaded.


## 16. Release parser and memory control

Record boundaries:
- VCV XML: `VariationArchive`
- RCV XML: `ClinVarSet`

Each completed record is parsed, normalized, written to bounded buffers, and cleared from the XML tree immediately.

In [15]:
def release_partition_dir(release_month, data_model, source_format):
    return (
        DATA_DIR
        / f"release_month={release_month}"
        / f"data_model={data_model}"
        / f"format={source_format}"
    )

def remove_incomplete_partition(release_dir: Path):
    if not release_dir.exists():
        return

    complete = release_dir / "release_complete.json"
    if complete.exists():
        return

    if DELETE_INCOMPLETE_PARTITION_BEFORE_RETRY:
        shutil.rmtree(release_dir)

def parse_stream_source(row: pd.Series, record_limit=None):
    release_month = str(row["release_month"])
    data_model = str(row["data_model"]).upper()
    source_format = str(row["format_generation"]).lower()
    source_url = str(row["url"])
    expected_md5 = source_expected_md5(row)

    release_dir = release_partition_dir(
        release_month, data_model, source_format
    )
    complete_path = release_dir / "release_complete.json"

    if (
        RESUME_COMPLETED_RELEASES
        and record_limit is None
        and complete_path.exists()
    ):
        print("SKIP completed:", release_month, data_model, source_format)
        return json.loads(complete_path.read_text())

    remove_incomplete_partition(release_dir)
    release_dir.mkdir(parents=True, exist_ok=True)

    writers = make_writers(release_dir)

    record_tag = "VariationArchive" if data_model == "VCV" else "ClinVarSet"

    counters = Counter()
    started = time.time()
    stream_complete = False
    parse_error = None
    computed_md5 = None
    computed_sha256 = None
    compressed_bytes_read = 0

    response = None

    try:
        response = SESSION.get(
            source_url,
            stream=True,
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        response.raw.decode_content = False

        hashing_raw = HashingReader(response.raw)

        with gzip.GzipFile(fileobj=hashing_raw, mode="rb") as gz:
            context = etree.iterparse(
                gz,
                events=("end",),
                recover=True,
                huge_tree=True,
            )

            for _, elem in context:
                if localname(elem.tag) != record_tag:
                    continue

                if data_model == "VCV":
                    row_vcv, links, scvs = parse_vcv_record(
                        elem=elem,
                        release_month=release_month,
                        source_url=source_url,
                        source_format=source_format,
                    )

                    if row_vcv is not None:
                        writers["vcv_state"].add(row_vcv)
                        writers["vcv_rcv_link"].add_many(links)
                        writers["scv_state"].add_many(scvs)

                        counters["aggregate_records"] += 1
                        counters["scv_records"] += len(scvs)
                        counters["vcv_rcv_links"] += len(links)

                else:
                    row_rcv, scvs = parse_rcv_record(
                        elem=elem,
                        release_month=release_month,
                        source_url=source_url,
                        source_format=source_format,
                    )

                    if row_rcv is not None:
                        writers["rcv_state"].add(row_rcv)
                        writers["scv_state"].add_many(scvs)

                        counters["aggregate_records"] += 1
                        counters["scv_records"] += len(scvs)

                clear_element(elem)

                if (
                    record_limit is not None
                    and counters["aggregate_records"] >= record_limit
                ):
                    counters["pilot_early_stop"] = 1
                    break

                if counters["aggregate_records"] % 100_000 == 0:
                    elapsed = max(time.time() - started, 1)
                    print(
                        f"{release_month} {data_model}: "
                        f"{counters['aggregate_records']:,} aggregates | "
                        f"{counters['scv_records']:,} SCVs | "
                        f"{counters['aggregate_records']/elapsed:,.0f} records/s"
                    )

        for w in writers.values():
            w.close()

        computed_md5 = hashing_raw.md5.hexdigest()
        computed_sha256 = hashing_raw.sha256.hexdigest()
        compressed_bytes_read = hashing_raw.bytes_read

        stream_complete = record_limit is None

    except Exception as e:
        parse_error = repr(e)
        for w in writers.values():
            try:
                w.close()
            except Exception:
                pass
        raise

    finally:
        if response is not None:
            response.close()

    elapsed = time.time() - started

    qc = {
        "release_month": release_month,
        "data_model": data_model,
        "source_format": source_format,
        "source_url": source_url,
        "expected_ncbi_md5": expected_md5,
        "computed_stream_md5": computed_md5,
        "computed_stream_sha256": computed_sha256,
        "compressed_bytes_read": int(compressed_bytes_read),
        "stream_complete": bool(stream_complete),
        "record_limit": record_limit,
        "aggregate_records": int(counters["aggregate_records"]),
        "scv_records": int(counters["scv_records"]),
        "vcv_rcv_links": int(counters["vcv_rcv_links"]),
        "elapsed_seconds": float(elapsed),
        "parse_error": parse_error,
        "md5_verification_status": (
            "verified"
            if stream_complete
            and expected_md5
            and computed_md5 == expected_md5
            else (
                "mismatch"
                if stream_complete
                and expected_md5
                and computed_md5 != expected_md5
                else (
                    "partial_stream"
                    if not stream_complete
                    else "expected_md5_unavailable"
                )
            )
        ),
        "parquet_rows": {
            name: int(w.total_rows)
            for name, w in writers.items()
        },
        "completed_utc": datetime.now(timezone.utc).isoformat(),
    }

    qc_path = release_dir / "release_qc.json"
    qc_path.write_text(json.dumps(qc, indent=2), encoding="utf-8")

    if stream_complete:
        if expected_md5 and computed_md5 != expected_md5:
            raise AssertionError(
                f"MD5 mismatch for {release_month} {data_model}: "
                f"expected {expected_md5}, got {computed_md5}"
            )

        complete_path.write_text(
            json.dumps(qc, indent=2),
            encoding="utf-8",
        )

    return qc

print("Streaming release parser loaded.")

Streaming release parser loaded.


## 17. Source-level retry wrapper

A source is retried from the beginning after transient network/parser failure. Incomplete Parquet shards are removed before retry so a failed source cannot silently duplicate records.

In [16]:
def process_source_with_retry(row, record_limit=None):
    last_error = None

    for attempt in range(1, MAX_SOURCE_ATTEMPTS + 1):
        try:
            print(
                f"\n=== {row['release_month']} | "
                f"{row['data_model']} | {row['format_generation']} | "
                f"attempt {attempt}/{MAX_SOURCE_ATTEMPTS} ==="
            )
            return parse_stream_source(row, record_limit=record_limit)

        except Exception as e:
            last_error = e
            print("Source attempt failed:", repr(e))

            if attempt < MAX_SOURCE_ATTEMPTS:
                wait = RETRY_BACKOFF_SECONDS * attempt
                print(f"Retrying in {wait}s ...")
                time.sleep(wait)

    raise RuntimeError(
        f"Source failed after {MAX_SOURCE_ATTEMPTS} attempts: "
        f"{row['release_month']} {row['data_model']} — {last_error!r}"
    )

## 18. Execute selected normalization run

For the pilot, this may take several minutes to tens of minutes depending on NCBI throughput and the density of SCVs.

A production full-month source can take substantially longer. The notebook processes sources **sequentially** to limit RAM pressure and make provenance/QC easy to audit.

In [17]:
run_results = []
run_failures = []

for _, source_row in selected.iterrows():
    try:
        result = process_source_with_retry(
            source_row,
            record_limit=record_limit,
        )
        run_results.append(result)

    except Exception as e:
        run_failures.append({
            "release_month": source_row["release_month"],
            "data_model": source_row["data_model"],
            "format_generation": source_row["format_generation"],
            "url": source_row["url"],
            "error": repr(e),
        })
        print("FINAL SOURCE FAILURE:", run_failures[-1])

results_df = pd.DataFrame(run_results)
failures_df = pd.DataFrame(run_failures)

display(results_df)

if len(failures_df):
    print("\nFailures:")
    display(failures_df)


=== 2023-01 | RCV | legacy | attempt 1/3 ===

=== 2023-01 | VCV | legacy | attempt 1/3 ===

=== 2026-08 | RCV | current | attempt 1/3 ===

=== 2026-08 | VCV | current | attempt 1/3 ===


,release_month,data_model,source_format,source_url,expected_ncbi_md5,computed_stream_md5,computed_stream_sha256,compressed_bytes_read,stream_complete,record_limit,aggregate_records,scv_records,vcv_rcv_links,elapsed_seconds,parse_error,md5_verification_status,parquet_rows,completed_utc
0,2023-01,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,d00be8862bbbd1d5a8b991c30246d224,fcf44e5eff1bf2ef9f069147d90c2db2,cb835237c95ead367fa74b0c3fa2f187ff1fee9dfe245a...,64487475,False,50000,50000,52610,0,55.141467,None,partial_stream,"{'vcv_state': 0, 'rcv_state': 50000, 'scv_stat...",2026-08-16T17:25:41.441136+00:00
1,2023-01,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,97acfb6a750b6a054c9800ac9f9a9924,263246392830f06419a754b250f82d70,d9299bd37ad962ef08af207381e63a15fb26f3cc271746...,70647864,False,50000,50000,62203,58942,57.903334,None,partial_stream,"{'vcv_state': 50000, 'rcv_state': 0, 'scv_stat...",2026-08-16T17:26:39.347228+00:00
2,2026-08,RCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,f6d225ba092e3d79b68dfaf06f6deb67,7517b80510ddabb0d3e157e9805905a4,7895c3f42a7a14e00675ed46a70b5c4fdb669724d425a3...,40894474,False,50000,50000,50441,0,40.962225,None,partial_stream,"{'vcv_state': 0, 'rcv_state': 50000, 'scv_stat...",2026-08-16T17:27:20.311338+00:00
3,2026-08,VCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,15eb690d6f3c9845373e53f7e9ad83f0,266b821d0f9285bc13bda70dd8bbed6b,7361b1ef76524bc4e00b55296ab363b984241f7fbce576...,42205226,False,50000,50000,54508,51512,42.840701,None,partial_stream,"{'vcv_state': 50000, 'rcv_state': 0, 'scv_stat...",2026-08-16T17:28:03.153450+00:00


## 19. Normalized artifact QC

These checks are intentionally structural:
- aggregate accessions must be present;
- SCV accessions must start with `SCV`;
- release and source lineage must be populated;
- production full parses must have verified MD5 when the expected NCBI checksum is available;
- current-format aggregate classifications remain axis-separated;
- legacy aggregate fields remain in legacy columns.

The notebook does **not** force a specific scientific event prevalence or germline fraction.

In [18]:
def read_partition_table(release_dir: Path, table_name: str) -> pd.DataFrame:
    paths = sorted((release_dir / table_name).glob("*.parquet"))
    if not paths:
        return pd.DataFrame(columns=TABLE_COLUMNS[table_name])
    return pd.concat(
        [pd.read_parquet(p) for p in paths],
        ignore_index=True,
    )

qc_rows = []

for result in run_results:
    release_dir = release_partition_dir(
        result["release_month"],
        result["data_model"],
        result["source_format"],
    )

    if result["data_model"] == "VCV":
        agg = read_partition_table(release_dir, "vcv_state")
        accession_col = "vcv_accession"
        prefix = "VCV"
    else:
        agg = read_partition_table(release_dir, "rcv_state")
        accession_col = "rcv_accession"
        prefix = "RCV"

    scv = read_partition_table(release_dir, "scv_state")

    accession_ok = (
        agg[accession_col].fillna("").str.startswith(prefix).all()
        if len(agg)
        else False
    )
    lineage_ok = (
        agg["release_month"].notna().all()
        and agg["source_url"].notna().all()
        if len(agg)
        else False
    )
    scv_accession_ok = (
        scv["scv_accession"].fillna("").str.startswith("SCV").all()
        if len(scv)
        else True
    )

    production_hash_ok = True
    if result["stream_complete"] and result["expected_ncbi_md5"]:
        production_hash_ok = (
            result["md5_verification_status"] == "verified"
        )

    qc_rows.append({
        "release_month": result["release_month"],
        "data_model": result["data_model"],
        "source_format": result["source_format"],
        "aggregate_rows": len(agg),
        "scv_rows": len(scv),
        "accession_ok": accession_ok,
        "lineage_ok": lineage_ok,
        "scv_accession_ok": scv_accession_ok,
        "production_hash_ok": production_hash_ok,
    })

artifact_qc = pd.DataFrame(qc_rows)
display(artifact_qc)

structural_cols = [
    "accession_ok",
    "lineage_ok",
    "scv_accession_ok",
    "production_hash_ok",
]

all_structural_ok = (
    artifact_qc[structural_cols].all(axis=None)
    if len(artifact_qc)
    else False
)

print("ALL STRUCTURAL ARTIFACT QC PASSED:", bool(all_structural_ok))

,release_month,data_model,source_format,aggregate_rows,scv_rows,accession_ok,lineage_ok,scv_accession_ok,production_hash_ok
0,2023-01,RCV,legacy,50000,52610,True,True,True,True
1,2023-01,VCV,legacy,50000,62203,True,True,True,True
2,2026-08,RCV,current,50000,50441,True,True,True,True
3,2026-08,VCV,current,50000,54508,True,True,True,True


ALL STRUCTURAL ARTIFACT QC PASSED: True


## 20. Classification-axis audit

This is a key protection against accidentally turning legacy ClinVar semantics into false modern labels.

For current-format rows we summarize modern classification axes separately.  
For legacy rows we summarize `legacy_classification_description` and the conservative term-based `legacy_germline_candidate` flag.

In [19]:
classification_audit_rows = []

for result in run_results:
    release_dir = release_partition_dir(
        result["release_month"],
        result["data_model"],
        result["source_format"],
    )

    table_name = (
        "vcv_state" if result["data_model"] == "VCV"
        else "rcv_state"
    )
    df = read_partition_table(release_dir, table_name)

    if df.empty:
        continue

    classification_audit_rows.append({
        "release_month": result["release_month"],
        "data_model": result["data_model"],
        "source_format": result["source_format"],
        "rows": len(df),
        "germline_nonnull": int(df["germline_description"].notna().sum()),
        "somatic_impact_nonnull": int(
            df["somatic_clinical_impact_description"].notna().sum()
        ),
        "oncogenicity_nonnull": int(
            df["oncogenicity_description"].notna().sum()
        ),
        "legacy_nonnull": int(
            df["legacy_classification_description"].notna().sum()
        ),
        "legacy_germline_candidate": int(
            df["legacy_germline_candidate"].fillna(False).sum()
        ),
    })

classification_audit = pd.DataFrame(classification_audit_rows)
display(classification_audit)

,release_month,data_model,source_format,rows,germline_nonnull,somatic_impact_nonnull,oncogenicity_nonnull,legacy_nonnull,legacy_germline_candidate
0,2023-01,RCV,legacy,50000,0,0,0,50000,49365
1,2023-01,VCV,legacy,50000,0,0,0,49883,47985
2,2026-08,RCV,current,50000,50000,0,0,0,0
3,2026-08,VCV,current,50000,50000,0,0,0,0


## 21. RCV–SCV relationship audit

The later temporal evidence graph depends on knowing which submitter assertions underlie each RCV state. This audit confirms that RCV-derived SCV rows preserve the parent RCV accession.

In [20]:
relationship_rows = []

for result in run_results:
    if result["data_model"] != "RCV":
        continue

    release_dir = release_partition_dir(
        result["release_month"],
        result["data_model"],
        result["source_format"],
    )
    scv = read_partition_table(release_dir, "scv_state")

    relationship_rows.append({
        "release_month": result["release_month"],
        "source_format": result["source_format"],
        "scv_rows": len(scv),
        "scv_with_parent_rcv": int(
            scv["parent_rcv_accession"].notna().sum()
        ) if len(scv) else 0,
        "unique_parent_rcv": int(
            scv["parent_rcv_accession"].nunique(dropna=True)
        ) if len(scv) else 0,
        "unique_submitters": int(
            scv["submitter_name"].nunique(dropna=True)
        ) if len(scv) else 0,
    })

relationship_audit = pd.DataFrame(relationship_rows)
display(relationship_audit)

,release_month,source_format,scv_rows,scv_with_parent_rcv,unique_parent_rcv,unique_submitters
0,2023-01,legacy,52610,52610,50000,544
1,2026-08,current,50441,50441,50000,331


## 22. Data dictionary

Freeze a compact data dictionary now so later feature notebooks do not reinterpret columns ad hoc.

In [21]:
data_dictionary = {
    "vcv_state": {
        "unit": "VCV aggregate state in one ClinVar monthly release",
        "identity": ["release_month", "vcv_accession"],
        "notes": (
            "Current classification axes are separate. Legacy single "
            "classification remains explicitly legacy."
        ),
    },
    "rcv_state": {
        "unit": "RCV variant-condition aggregate state in one monthly release",
        "identity": ["release_month", "rcv_accession"],
        "notes": (
            "Primary GES 3.0 longitudinal prediction-unit foundation."
        ),
    },
    "scv_state": {
        "unit": "Submitted ClinVar assertion visible in one monthly release",
        "identity": [
            "release_month",
            "source_data_model",
            "scv_accession",
            "parent_rcv_accession",
            "parent_vcv_accession",
        ],
        "notes": (
            "RCV-derived SCV rows preserve condition-context relationship. "
            "VCV-derived SCVs may overlap the same SCV accession and must not "
            "be naively double-counted downstream."
        ),
    },
    "vcv_rcv_link": {
        "unit": "VCV-to-RCV link visible in one monthly VCV release",
        "identity": [
            "release_month",
            "vcv_accession",
            "rcv_accession",
        ],
    },
    "classification_policy": {
        "current": (
            "Preserve germline, somatic clinical impact, and oncogenicity "
            "as separate fields."
        ),
        "legacy": (
            "Preserve single classification as legacy_*; "
            "legacy_germline_candidate is only a conservative term flag, "
            "not final cohort eligibility."
        ),
    },
}

data_dictionary_path = META_DIR / "stage01_data_dictionary.json"
data_dictionary_path.write_text(
    json.dumps(data_dictionary, indent=2),
    encoding="utf-8",
)

print(json.dumps(data_dictionary, indent=2))

{
  "vcv_state": {
    "unit": "VCV aggregate state in one ClinVar monthly release",
    "identity": [
      "release_month",
      "vcv_accession"
    ],
    "notes": "Current classification axes are separate. Legacy single classification remains explicitly legacy."
  },
  "rcv_state": {
    "unit": "RCV variant-condition aggregate state in one monthly release",
    "identity": [
      "release_month",
      "rcv_accession"
    ],
    "notes": "Primary GES 3.0 longitudinal prediction-unit foundation."
  },
  "scv_state": {
    "unit": "Submitted ClinVar assertion visible in one monthly release",
    "identity": [
      "release_month",
      "source_data_model",
      "scv_accession",
      "parent_rcv_accession",
      "parent_vcv_accession"
    ],
    "notes": "RCV-derived SCV rows preserve condition-context relationship. VCV-derived SCVs may overlap the same SCV accession and must not be naively double-counted downstream."
  },
  "vcv_rcv_link": {
    "unit": "VCV-to-RCV link visib

## 23. Freeze run summary and artifact hashes

Each small metadata/QC artifact receives a SHA-256. Individual Parquet shards also receive hashes so later stages can detect silent changes.

In [22]:
def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

# Save run tables.
results_path = QC_DIR / f"stage01_results_{RUN_PROFILE}_{RUN_UTC[:10]}.csv"
failures_path = QC_DIR / f"stage01_failures_{RUN_PROFILE}_{RUN_UTC[:10]}.csv"
artifact_qc_path = QC_DIR / f"stage01_artifact_qc_{RUN_PROFILE}_{RUN_UTC[:10]}.csv"
class_qc_path = QC_DIR / f"stage01_classification_audit_{RUN_PROFILE}_{RUN_UTC[:10]}.csv"
relation_qc_path = QC_DIR / f"stage01_rcv_scv_audit_{RUN_PROFILE}_{RUN_UTC[:10]}.csv"

results_df.to_csv(results_path, index=False)
failures_df.to_csv(failures_path, index=False)
artifact_qc.to_csv(artifact_qc_path, index=False)
classification_audit.to_csv(class_qc_path, index=False)
relationship_audit.to_csv(relation_qc_path, index=False)

runtime_metadata = {
    "run_utc": RUN_UTC,
    "run_profile": RUN_PROFILE,
    "manifest_origin": manifest_origin,
    "record_limit_per_source": record_limit,
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "pyarrow": pa.__version__,
    "lxml": lxml.__version__,
    "requests": requests.__version__,
    "stream_remote_gzip": STREAM_REMOTE_GZIP,
    "selected_sources": int(len(selected)),
    "successful_sources": int(len(run_results)),
    "failed_sources": int(len(run_failures)),
    "all_structural_qc_passed": bool(all_structural_ok),
}

runtime_path = META_DIR / "stage01_runtime_metadata.json"
runtime_path.write_text(
    json.dumps(runtime_metadata, indent=2),
    encoding="utf-8",
)

hash_rows = []
for base in [META_DIR, QC_DIR, DATA_DIR]:
    for p in base.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in {".json", ".csv", ".parquet"}:
            continue
        hash_rows.append({
            "relative_path": str(p.relative_to(STAGE01_DIR)),
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

hash_df = pd.DataFrame(hash_rows).sort_values("relative_path")
hash_path = META_DIR / "stage01_artifact_sha256.csv"
hash_df.to_csv(hash_path, index=False)

print(json.dumps(runtime_metadata, indent=2))
print("\nHashed artifacts:", len(hash_df))
display(hash_df.head(20))

{
  "run_utc": "2026-08-16T17:24:41.113423+00:00",
  "run_profile": "PILOT_SCHEMA_BRIDGE",
  "manifest_origin": "reconstructed_live_for_pilot_only",
  "record_limit_per_source": 50000,
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "pandas": "2.2.2",
  "numpy": "2.0.2",
  "pyarrow": "18.1.0",
  "lxml": "6.1.1",
  "requests": "2.32.4",
  "stream_remote_gzip": true,
  "selected_sources": 4,
  "successful_sources": 4,
  "failed_sources": 0,
  "all_structural_qc_passed": true
}

Hashed artifacts: 28


,relative_path,bytes,sha256
0,metadata/stage01_data_dictionary.json,1412,01835d293e0c7d04eba25f0c056df65c44cfe06bb86378...
1,metadata/stage01_run_manifest_PILOT_SCHEMA_BRI...,1427,0fb6babc9ccee9db138cc058c34a2961caa0645f2a7d12...
2,metadata/stage01_runtime_metadata.json,553,63ef75ca5f08b1926277263ad16ac7aa2e011224fcd067...
17,normalized/release_month=2023-01/data_model=RC...,1287830,a6fdab7762b1e495cf6e84c16d57db4cbb7cba30371b77...
14,normalized/release_month=2023-01/data_model=RC...,877,e4232984c7148113749b2cc768bafc6723fac19d90ebbf...
15,normalized/release_month=2023-01/data_model=RC...,1746300,034761bfd561943c9b56f7f6a6de4ffd03108b1fa566c1...
16,normalized/release_month=2023-01/data_model=RC...,99876,31146c4770d53ee910f75a0fca652b56a607f7c4bde85d...
8,normalized/release_month=2023-01/data_model=VC...,889,44b4a437024e452151d922efb2598d484f9f4b63279234...
9,normalized/release_month=2023-01/data_model=VC...,1628117,a6d5d2585c5c703a104d45811f3b33c05ddf41b68d33a2...
10,normalized/release_month=2023-01/data_model=VC...,263276,d302bcc530a8f79ce1100738bba829433c6449b7cae853...


## 24. Go / no-go decision for the next stage

### Pilot go criterion
Proceed from the pilot to production batches only if:
1. legacy and current sources both produced aggregate records;
2. accession and lineage QC passed;
3. RCV SCVs preserve parent RCV identity;
4. current-format classification axes appear in current fields;
5. legacy classifications appear in `legacy_*` fields rather than being silently forced into modern axes.

### Production go criterion
A source is production-complete only if:
- the full gzip stream was consumed;
- the NCBI MD5 matched when an expected MD5 was available;
- structural artifact QC passed;
- a `release_complete.json` marker exists.

### Important
A **pilot partial stream is not a frozen research dataset**. It validates parser behavior only.

In [23]:
decision = {
    "run_profile": RUN_PROFILE,
    "successful_sources": len(run_results),
    "failed_sources": len(run_failures),
    "structural_qc_passed": bool(all_structural_ok),
}

if RUN_PROFILE == "PILOT_SCHEMA_BRIDGE":
    formats_seen = set(results_df.get("source_format", pd.Series(dtype=str)))
    models_seen = set(results_df.get("data_model", pd.Series(dtype=str)))

    decision["legacy_seen"] = "legacy" in formats_seen
    decision["current_seen"] = "current" in formats_seen
    decision["vcv_seen"] = "VCV" in models_seen
    decision["rcv_seen"] = "RCV" in models_seen

    decision["pilot_go_for_production"] = bool(
        all_structural_ok
        and len(run_failures) == 0
        and decision["legacy_seen"]
        and decision["current_seen"]
        and decision["vcv_seen"]
        and decision["rcv_seen"]
    )

else:
    full_verified = True
    for r in run_results:
        if r["expected_ncbi_md5"]:
            full_verified = full_verified and (
                r["md5_verification_status"] == "verified"
            )

    decision["all_expected_md5_verified"] = bool(full_verified)
    decision["batch_go_for_stage02"] = bool(
        all_structural_ok
        and len(run_failures) == 0
        and full_verified
    )

print(json.dumps(decision, indent=2))

{
  "run_profile": "PILOT_SCHEMA_BRIDGE",
  "successful_sources": 4,
  "failed_sources": 0,
  "structural_qc_passed": true,
  "legacy_seen": true,
  "current_seen": true,
  "vcv_seen": true,
  "rcv_seen": true,
  "pilot_go_for_production": true
}


## 25. Create a compact Stage 01 metadata/QC bundle

The full normalized Parquet lake can be too large to ZIP in Colab. This bundle intentionally contains only the run manifest, metadata, QC tables, data dictionary, and hash manifest.

The Parquet partitions remain in `stage01/normalized/`.

In [24]:
bundle_dir = Path("/content/GES3_STAGE01_METADATA_BUNDLE")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True)

for p in [run_manifest_path, data_dictionary_path, runtime_path, hash_path]:
    if Path(p).exists():
        shutil.copy2(p, bundle_dir / Path(p).name)

for p in QC_DIR.glob("*"):
    if p.is_file():
        shutil.copy2(p, bundle_dir / p.name)

bundle_zip = shutil.make_archive(
    "/content/GES3_STAGE01_METADATA_QC",
    "zip",
    root_dir=bundle_dir,
)

print("Metadata/QC bundle:", bundle_zip)
print("Bundle bytes:", Path(bundle_zip).stat().st_size)

Metadata/QC bundle: /content/GES3_STAGE01_METADATA_QC.zip
Bundle bytes: 5716


## 26. Preview normalized rows

This is a small inspection view only. It is not used to tune scientific labels or thresholds.

In [25]:
for result in run_results[:4]:
    release_dir = release_partition_dir(
        result["release_month"],
        result["data_model"],
        result["source_format"],
    )

    table_name = (
        "vcv_state" if result["data_model"] == "VCV"
        else "rcv_state"
    )

    df = read_partition_table(release_dir, table_name)

    print(
        "\n",
        result["release_month"],
        result["data_model"],
        result["source_format"],
        table_name,
    )
    display(df.head(5))


 2023-01 RCV legacy rcv_state


,release_month,source_url,source_format,rcv_accession,rcv_version,vcv_accession,vcv_version,variation_id,variation_name,germline_description,...,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,condition_names_json,condition_ids_json,gene_symbols_json,citation_ids_json,scv_count
0,2023-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000000002,3,VCV000018391,1,18391,None,None,...,None,Pathogenic,no assertion criteria provided,2014-12-01,True,"[""Spastic ataxia 4""]","[""MONDO:MONDO:0013354"", ""Genetic Alliance:Atax...",[],"[""PubMed:20970105"", ""PubMed:25008111""]",1
1,2023-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000000006,5,VCV000018395,2,18395,None,None,...,None,Pathogenic,no assertion criteria provided,2010-01-01,True,"[""DUFFY BLOOD GROUP SYSTEM, FY(a-b-) PHENOTYPE""]","[""OMIM:613665.0002"", ""OMIM:613665.0004""]",[],"[""PubMed:7663520"", ""PubMed:8651934"", ""PubMed:2...",1
2,2023-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000000007,4,VCV000018395,2,18395,None,None,...,None,protective,no assertion criteria provided,2017-12-11,True,"[""Plasmodium vivax, resistance to""]","[""MedGen:C1970105""]",[],"[""PubMed:7663520"", ""PubMed:8651934"", ""PubMed:2...",1
3,2023-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000000008,4,VCV000018395,2,18395,None,None,...,None,association,no assertion criteria provided,2010-01-01,True,"[""White blood cell count quantitative trait lo...","[""Genetic Alliance:White+blood+cell+count+quan...",[],"[""PubMed:7663520"", ""PubMed:8651934"", ""PubMed:2...",1
4,2023-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000000009,4,VCV000018396,1,18396,None,None,...,None,Pathogenic,no assertion criteria provided,2001-06-01,True,"[""DUFFY BLOOD GROUP SYSTEM, FY(bwk) PHENOTYPE""]","[""OMIM:613665.0003""]",[],"[""PubMed:9731074"", ""PubMed:9886340"", ""PubMed:8...",1



 2023-01 VCV legacy vcv_state


,release_month,source_url,source_format,vcv_accession,vcv_version,variation_id,variation_type,variation_name,date_created,date_last_updated,...,oncogenicity_description,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,gene_symbols_json,rcv_accessions_json,citation_ids_json,scv_count
0,2023-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000830334,1,830334,single nucleotide variant,NM_000769.1(CYP2C19):c.680C>T (p.Pro227Leu),2019-06-17,2020-10-31,...,None,None,None,None,None,False,"[""CYP2C19""]",[],[],0
1,2023-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000830335,1,830335,single nucleotide variant,NM_000769.4(CYP2C19):c.766G>A (p.Asp256Asn),2019-06-17,2020-10-31,...,None,None,None,None,None,False,"[""CYP2C19""]",[],[],0
2,2023-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000830336,1,830336,single nucleotide variant,NM_000769.4(CYP2C19):c.151A>G (p.Ser51Gly),2019-06-17,2020-10-31,...,None,None,None,None,None,False,"[""CYP2C19""]",[],[],0
3,2023-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000830337,1,830337,single nucleotide variant,NM_000769.4(CYP2C19):c.-1041G>A,2019-06-17,2020-10-31,...,None,None,None,None,None,False,"[""CYP2C19"", ""LOC110599570""]",[],[],0
4,2023-01,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000830338,1,830338,Variation,CYP2C19*1,2019-06-17,2020-10-31,...,None,None,None,None,None,False,"[""CYP2C19"", ""LOC110599570""]",[],[],0



 2026-08 RCV current rcv_state


,release_month,source_url,source_format,rcv_accession,rcv_version,vcv_accession,vcv_version,variation_id,variation_name,germline_description,...,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,condition_names_json,condition_ids_json,gene_symbols_json,citation_ids_json,scv_count
0,2026-08,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,current,RCV000002751,4,VCV000002633,1,2633,None,Pathogenic,...,None,None,None,None,False,"[""Nephronophthisis 3""]","[""GeneTests:72010"", ""MONDO:MONDO:0011456"", ""Ge...",[],"[""PubMed:12872122"", ""PubMed:27336129"", ""BookSh...",1
1,2026-08,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,current,RCV000002738,3,VCV000002620,1,2620,None,drug response,...,None,None,None,None,False,"[""Ezetimibe response""]","[""OMIM:608010.0001"", ""OMIM:608010.0002"", ""MedG...",[],"[""PubMed:15679830"", ""https://dailymed.nlm.nih....",1
2,2026-08,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,current,RCV000002603,5,VCV000002497,2,2497,None,Pathogenic,...,None,None,None,None,False,"[""Chondrosarcoma""]","[""Genetic Alliance:Chondrosarcoma/1378"", ""Huma...",[],"[""PubMed:8981950"", ""https://www.ncbi.nlm.nih.g...",1
3,2026-08,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,current,RCV000002558,3,VCV000002454,1,2454,None,Pathogenic,...,None,None,None,None,False,"[""Sialidosis type 1""]","[""MONDO:MONDO:0019346"", ""Office of Rare Diseas...",[],"[""PubMed:11829139""]",1
4,2026-08,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,current,RCV000019874,3,VCV000018214,3,18214,None,other,...,None,None,None,None,False,"[""PROALBUMIN JAFFNA""]","[""OMIM:103600.0031""]",[],"[""PubMed:2792379""]",1



 2026-08 VCV current vcv_state


,release_month,source_url,source_format,vcv_accession,vcv_version,variation_id,variation_type,variation_name,date_created,date_last_updated,...,oncogenicity_description,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,gene_symbols_json,rcv_accessions_json,citation_ids_json,scv_count
0,2026-08,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,current,VCV000000455,2,455,Deletion,NM_000097.7(CPOX):c.489_509del (p.Cys164_Val17...,2020-05-31,2022-04-25,...,None,None,None,None,None,False,"[""CPOX""]","[""RCV000000484""]","[""PubMed:8990017""]",1
1,2026-08,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,current,VCV000000441,1,441,Deletion,NG_007480.1:g.(7267_8082)_(8469_9169)del,2015-08-22,2023-01-07,...,None,None,None,None,None,False,"[""BCAM""]","[""RCV000000470""]","[""PubMed:17319831""]",1
2,2026-08,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,current,VCV000000838,2,838,Deletion,"APC, 1-BP DEL, 3720T",2013-04-04,2022-12-17,...,None,None,None,None,None,False,"[""APC""]","[""RCV000000881""]","[""PubMed:10782927""]",1
3,2026-08,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,current,VCV000000984,2,984,Indel,NM_024782.3(NHEJ1):c.177+1_177+3delinsTT,2013-09-21,2024-09-01,...,None,None,None,None,None,False,"[""NHEJ1""]","[""RCV000001035""]","[""PubMed:16439204""]",1
4,2026-08,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,current,VCV000001172,1,1172,Duplication,NC_000008.11:g.(142876886_142877022)_(14291490...,2013-04-04,2024-05-01,...,None,None,None,None,None,False,"[""CYP11B1"", ""CYP11B2"", ""LOC106799833"", ""LOC106...","[""RCV000001231""]","[""PubMed:15324322"", ""PubMed:1731223""]",1


## 27. Stage 01 completion status

### After the default pilot
If `pilot_go_for_production = true`, rerun this notebook using **`PRODUCTION_BATCH`** and a small month range. Keep the frozen Stage 00 manifest unchanged.

Suggested first production batches:
- `2021-01` → `2021-03`
- `2021-04` → `2021-06`
- continue in small resumable blocks

Once the full longitudinal window has been normalized, Stage 02 should consume only release partitions with successful completion markers.

### Next notebook
**GES 3.0 — Notebook 02: `temporal_identifier_linkage`**

Stage 02 will:
1. link VCV and RCV identities across monthly releases;
2. link RCV-month states to underlying SCV versions;
3. distinguish true disappearance from source/parser failure;
4. represent entry, update, persistence, and censoring;
5. calculate linkage stability;
6. generate the first longitudinal RCV trajectories;
7. freeze cross-release identity rules before event labeling.

### Scientific boundary
Do not calculate GES-R, survival targets, or future-instability labels yet.

The workflow remains:

**archive → normalize → link → define transitions → audit leakage → build dynamic features → model.**